In [1]:
# ============================================================
# CodeAlpha Cloud Computing Internship — Task 2
# Detecting Data Leaks Using SQL Injection + AES-256 Encryption
# Single-file version for Google Colab (paste into one cell & run)
# ============================================================

!pip install cryptography -q

import os, re, base64, sqlite3, logging
from datetime import datetime
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
from cryptography.hazmat.primitives import hashes

# ---------------------------------------------------------
# 1. AES-256-GCM ENCRYPTION
# ---------------------------------------------------------
class AES256Cipher:
    def __init__(self, master_password: str, salt: bytes):
        kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=salt, iterations=200_000)
        self.key = kdf.derive(master_password.encode())
        self.aesgcm = AESGCM(self.key)

    def encrypt(self, plaintext: str) -> str:
        nonce = os.urandom(12)
        ct = self.aesgcm.encrypt(nonce, plaintext.encode(), None)
        return base64.b64encode(nonce + ct).decode()

    def decrypt(self, token: str) -> str:
        blob = base64.b64decode(token.encode())
        nonce, ct = blob[:12], blob[12:]
        return self.aesgcm.decrypt(nonce, ct, None).decode()


def make_cipher(master_password: str, salt: bytes = None) -> AES256Cipher:
    salt = salt or os.urandom(16)
    return AES256Cipher(master_password, salt), salt


# ---------------------------------------------------------
# 2. SQL INJECTION DETECTOR
# ---------------------------------------------------------
logging.basicConfig(filename="injection_attempts.log", level=logging.WARNING,
                     format="%(asctime)s | %(message)s")

SQLI_PATTERNS = [
    r"(\bOR\b|\bAND\b)\s+[\d\w'\"]+\s*=\s*[\d\w'\"]+",
    r"--",
    r";\s*(DROP|DELETE|UPDATE|INSERT|ALTER)\s",
    r"\bUNION\b\s+\bSELECT\b",
    r"\bSLEEP\s*\(",
    r"\bBENCHMARK\s*\(",
    r"'\s*OR\s*'1'\s*=\s*'1",
    r"xp_cmdshell",
    r"\bEXEC(UTE)?\b\s*\(",
    r"/\*.*\*/",
    r"\bWAITFOR\s+DELAY\b",
]
_compiled = [re.compile(p, re.IGNORECASE) for p in SQLI_PATTERNS]


class InjectionDetected(Exception):
    pass


def scan(user_input, source_field="unknown", user_ip="local") -> bool:
    if user_input is None:
        return True
    for pattern in _compiled:
        if pattern.search(str(user_input)):
            logging.warning(f"BLOCKED | field={source_field} | ip={user_ip} | "
                             f"pattern='{pattern.pattern}' | input='{user_input[:100]}'")
            raise InjectionDetected(f"Potential SQL injection detected in field '{source_field}'")
    return True


def scan_dict(fields: dict, user_ip="local") -> bool:
    for field_name, value in fields.items():
        scan(value, source_field=field_name, user_ip=user_ip)
    return True


# ---------------------------------------------------------
# 3. SECURE DB (parameterized queries + encryption + detector)
# ---------------------------------------------------------
class SecureDB:
    def __init__(self, db_path="secure_users.db", master_password="change_this_master_key"):
        self.conn = sqlite3.connect(db_path)
        self.cipher, self.salt = make_cipher(master_password)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                password_enc TEXT NOT NULL,
                email_enc TEXT NOT NULL,
                phone_enc TEXT,
                created_at TEXT NOT NULL
            )
        """)
        self.conn.commit()

    def register_user(self, username, password, email, phone=None, user_ip="local"):
        scan_dict({"username": username, "password": password, "email": email, "phone": phone}, user_ip=user_ip)
        password_enc = self.cipher.encrypt(password)
        email_enc = self.cipher.encrypt(email)
        phone_enc = self.cipher.encrypt(phone) if phone else None
        try:
            self.conn.execute(
                "INSERT INTO users (username, password_enc, email_enc, phone_enc, created_at) VALUES (?, ?, ?, ?, ?)",
                (username, password_enc, email_enc, phone_enc, datetime.utcnow().isoformat()))
            self.conn.commit()
            return True
        except sqlite3.IntegrityError:
            raise ValueError(f"Username '{username}' already exists")

    def get_user(self, username, user_ip="local"):
        scan_dict({"username": username}, user_ip=user_ip)
        cur = self.conn.execute(
            "SELECT id, username, password_enc, email_enc, phone_enc, created_at FROM users WHERE username = ?",
            (username,))
        row = cur.fetchone()
        if not row:
            return None
        return {"id": row[0], "username": row[1], "password": self.cipher.decrypt(row[2]),
                "email": self.cipher.decrypt(row[3]), "phone": self.cipher.decrypt(row[4]) if row[4] else None,
                "created_at": row[5]}

    def verify_login(self, username, password, user_ip="local"):
        scan_dict({"username": username, "password": password}, user_ip=user_ip)
        user = self.get_user(username, user_ip=user_ip)
        return bool(user and user["password"] == password)

    def close(self):
        self.conn.close()


# ---------------------------------------------------------
# 4. DEMO
# ---------------------------------------------------------
def banner(text):
    print("\n" + "=" * 60)
    print(text)
    print("=" * 60)

MASTER_KEY = "S3cure-Master-Key-Change-In-Prod"

if os.path.exists("secure_users.db"): os.remove("secure_users.db")

db = SecureDB(master_password=MASTER_KEY)

banner("1. Registering legitimate user (AES-256 encrypted storage)")
db.register_user("palak_dev", "MyP@ssw0rd!", "palak@example.com", "9999999999")
print("User 'palak_dev' registered successfully.")

banner("2. Reading back user (decrypted on the fly)")
print(db.get_user("palak_dev"))

banner("3. Valid login")
print("Login success:", db.verify_login("palak_dev", "MyP@ssw0rd!"))

banner("4. Invalid password")
print("Login success:", db.verify_login("palak_dev", "wrongpass"))

banner("5. Simulating SQL Injection attacks (BLOCKED + logged)")
attacks = [
    "admin' OR '1'='1",
    "'; DROP TABLE users; --",
    "' UNION SELECT username, password_enc FROM users --",
    "admin'--",
    "1' AND SLEEP(5)--",
]
for payload in attacks:
    try:
        db.verify_login(payload, "irrelevant", user_ip="203.0.113.42")
        print(f"[NOT DETECTED] {payload}")
    except InjectionDetected as e:
        print(f"[BLOCKED] {payload}  ->  {e}")

db.close()

/tmp/ipykernel_17361/3737939735.py:110: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  (username, password_enc, email_enc, phone_enc, datetime.utcnow().isoformat()))



1. Registering legitimate user (AES-256 encrypted storage)
User 'palak_dev' registered successfully.

2. Reading back user (decrypted on the fly)
{'id': 1, 'username': 'palak_dev', 'password': 'MyP@ssw0rd!', 'email': 'palak@example.com', 'phone': '9999999999', 'created_at': '2026-07-02T10:00:29.724296'}

3. Valid login
Login success: True

4. Invalid password
Login success: False

5. Simulating SQL Injection attacks (BLOCKED + logged)
[BLOCKED] admin' OR '1'='1  ->  Potential SQL injection detected in field 'username'
[BLOCKED] '; DROP TABLE users; --  ->  Potential SQL injection detected in field 'username'
[BLOCKED] ' UNION SELECT username, password_enc FROM users --  ->  Potential SQL injection detected in field 'username'
[BLOCKED] admin'--  ->  Potential SQL injection detected in field 'username'
[BLOCKED] 1' AND SLEEP(5)--  ->  Potential SQL injection detected in field 'username'
